# Dataset AIMD de BGDY 3×3 (LUMI + MareNostrum 5)

Este notebook lee las dinámicas NVT de Quantum ESPRESSO y prepara una selección de configuraciones para entrenar un potencial. **Las trayectorias estables y las metaestables se tratan por separado.**

### Cómo usarlo
1. Coloca el notebook en la carpeta que contiene `DATASET_AIMD_LUMI` y `DATASET_AIMD_MN5`, o cambia `DATA_ROOT` en la sección 1.
2. En la **sección 1** elige:
   * qué temperaturas usas de cada régimen (`T_ESTABLES`, `T_METAESTABLES`);
   * cuántas configuraciones estables quieres de cada temperatura (`CUOTA_ESTABLES`);
   * cada cuánto se toma una metaestable (`INTERVALO_META_FS`).
3. Ejecuta todo (*Run All*). La primera vez tarda unos 3 minutos; después, menos de 1, porque los pasos lentos se guardan en `WORK`.

### Qué hace

| § | Paso | Resultado |
|---|---|---|
| 1 | Parámetros | Rutas, temperaturas, cuotas e intervalos |
| 2 | Funciones | Herramientas que usan las demás secciones (no hace falta modificarlas) |
| 3 | Leer las simulaciones | Energía, fuerzas y posiciones de cada paso; reinicios unidos |
| 4 | Comprobaciones | Paso temporal y continuidad en los reinicios |
| 5 | Construir el dataset | Se quitan los primeros 350 fs y el final de las metaestables; solo entran las temperaturas elegidas |
| 6 | Termalización | Temperatura frente al tiempo |
| 7 | Fuerzas | Distribución de \|F\| |
| 8 | Descriptores y UMAP | Un vector SOAP por configuración y su mapa 2D |
| 9 | Mapas UMAP | Configuraciones por temperatura y por régimen |
| 10 | Configuraciones independientes | n_eff, como guía para elegir las cuotas |
| 11 | Selección | Estables por FPS (con tu cuota), metaestables cada 20 fs, y comprobación de redundancia entre temperaturas |
| 12 | Exportar | `extxyz` y CSV |

### Qué obtienes (en la carpeta `WORK`)
* `BGDY_3x3_seleccion_N<n>.extxyz`: las configuraciones elegidas, con energía, fuerzas y etiquetas. Las posiciones salen envueltas en la celda.
* `BGDY_3x3_indice_configuraciones.csv`: todas las configuraciones, con su posición en UMAP y su estado (`FPS`, `tiempo`, `redundante` o vacío).
* `figs/`: todas las figuras en PNG.

**Requisitos:** `pip install numpy pandas matplotlib ase dscribe scikit-learn umap-learn`

## 1. Parámetros
Lo que normalmente se cambia está aquí. Si modificas algo, vuelve a ejecutar desde la sección donde se usa (o todo).

In [ ]:
from pathlib import Path

# ---- Dónde están los datos y dónde se guardan los resultados -------------------------
DATA_ROOT = Path("./")             # carpeta que contiene DATASET_AIMD_LUMI y DATASET_AIMD_MN5
WORK      = Path("bgdy_work_2")    # resultados intermedios, figuras y dataset final

# ---- Qué trayectorias entran -----------------------------------------------------------
T_ESTABLES     = [500, 1000, 1500, 2000, 2500]   # temperaturas (K) de las que se usan las trayectorias ESTABLES
T_METAESTABLES = [2250, 2500, 2750]        # temperaturas (K) de las que se usan las trayectorias METAESTABLES

# ---- Limpieza de cada trayectoria (todas, estables y metaestables) ----------------------
T_EQ_PS               = 0.5       # ps que se descartan al inicio de cada trayectoria (equilibrado)
N_ULTIMOS_METAESTABLE = 2          # pasos MD que se quitan al final de cada trayectoria metaestable
F_MAX                 = 100.0      # eV/Å: se descarta toda configuración con algún |F| por encima

# ---- ESTABLES: FPS dentro de cada temperatura, con este número de configuraciones -------
CUOTA_ESTABLES = {500: 50, 1000: 100, 1500: 200, 2000: 250, 2500: 200}   # una entrada por cada T de T_ESTABLES

# ---- METAESTABLES: a intervalos fijos de tiempo -----------------------------------------
INTERVALO_META_FS     = 40         # una configuración cada este tiempo (fs)
DESCARTAR_ULTIMA_META = True       # quitar la última configuración elegida de cada trayectoria metaestable

# ---- Redundancia entre temperaturas (estables) ------------------------------------------
UMBRAL_REDUNDANCIA    = "fps"          # "fps" = resolución del FPS de cada temperatura; o un número (distancia SOAP)
DESCARTAR_REDUNDANTES = True           # True = quitar una de cada pareja redundante; False = solo informar
DESCARTAR_EN          = "mayor_cuota"  # de qué temperatura se quita: "mayor_cuota", "mayor_T" o "menor_T"

# ---- Exportación ---------------------------------------------------------------------
WRAP          = True               # envolver las posiciones dentro de la celda (QE las da sin envolver)
EXPORTAR_TODO = False              # True = escribir también un extxyz con TODO el dataset (≈ 300 MB)

# ---- Opciones técnicas (normalmente no hace falta tocarlas) ---------------------------
SOAP_PARAMS    = dict(r_cut=5.0, n_max=6, l_max=4, sigma=0.5)
PCA_VAR        = 0.99              # la PCA interna conserva este porcentaje de la varianza
UMAP_NEIGHBORS = 30
UMAP_MIN_DIST  = 0.1
N_JOBS         = 2                 # núcleos para SOAP
SEED           = 0
RECALCULAR     = False             # True = repetir los pasos lentos aunque estén guardados

## 2. Librerías y funciones
Aquí se cargan las librerías y se definen las funciones que usan las secciones siguientes. **Basta con ejecutar estas celdas; no hace falta leerlas para usar el notebook.**

In [ ]:
import re, time, pickle, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from IPython.display import display

warnings.filterwarnings("ignore", message=".*IProgress not found.*")
warnings.filterwarnings("ignore", message=".*n_jobs value.*overridden.*")
%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

WORK.mkdir(exist_ok=True)
(WORK / "figs").mkdir(exist_ok=True)
assert DATA_ROOT.exists(), f"No encuentro {DATA_ROOT.resolve()}"

# Unidades: pw.x trabaja en unidades atómicas de Rydberg
BOHR    = 0.529177210903       # Å
RY      = 13.605693122994      # eV
RY_BOHR = RY / BOHR            # Ry/bohr -> eV/Å (× 25.711)

# Estilo de las figuras
AZUL, NARANJA, GRIS, ROJO, TINTA = "#2a78d6", "#eb6834", "#d6d5ce", "#d03b3b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": TINTA,
    "xtick.color": "#898781", "ytick.color": "#898781",
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False,
})
HEAT = LinearSegmentedColormap.from_list("heat", plt.get_cmap("YlOrRd")(np.linspace(0.30, 1.0, 256)))

def guardar(fig, nombre):
    fig.savefig(WORK / "figs" / f"{nombre}.png")

def limpiar(ax):
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

**Lectura de los outputs de `pw.x`.** En cada paso MD, `pw.x` escribe: SCF → energía (`!`) → fuerzas → `Entering Dynamics: iteration = n` → posiciones nuevas. Por eso el SCF que precede a `iteration = n` corresponde al paso n − 1.

**Reinicios.** Los `output.1, .2, .3…` de una carpeta son la misma trayectoria y se unen por número de paso. La cabecera de `output.2, .3…` muestra la geometría inicial (t = 0), pero `pw.x` continúa desde la última geometría del output anterior. Las funciones lo corrigen.

In [ ]:
FLOAT = r"[-+]?\d*\.\d+(?:[eEdD][-+]?\d+)?"

def leer_output(path):
    # Lee un output de pw.x (MD): metadatos, especies, celda, frames (SCF convergidos) y temperatura.
    lines = Path(path).read_text(errors="replace").splitlines()
    nat = alat = dt_fs = None
    cell_alat, species, pos0 = [], [], []
    frames, thermo = [], []
    cur_pos = pending = last_it = None
    i, n = 0, len(lines)
    while i < n:
        L = lines[i]
        if nat is None and "number of atoms/cell" in L:
            nat = int(L.split("=")[1])
        elif alat is None and "lattice parameter (alat)" in L:
            alat = float(L.split("=")[1].split()[0])
        elif len(cell_alat) < 3 and re.match(r"\s+a\([123]\) = \(", L):
            cell_alat.append([float(x) for x in re.findall(FLOAT, L.split("=")[1])[:3]])
        elif dt_fs is None and "Time step" in L and "femto-seconds" in L:
            dt_fs = float(L.split(",")[1].split()[0])
        elif not pos0 and "positions (alat units)" in L:
            for k in range(1, nat + 1):
                species.append(lines[i + k].split()[1])
                pos0.append([float(x) for x in re.findall(FLOAT, lines[i + k].split("=")[1])[:3]])
            cur_pos = np.array(pos0) * alat * BOHR
            i += nat
        elif L.startswith("ATOMIC_POSITIONS"):
            cur_pos = np.array([[float(x) for x in lines[i + k].split()[1:4]] for k in range(1, nat + 1)])
            i += nat
        elif L.startswith("!"):
            pending = dict(pos=cur_pos.copy(), E=float(L.split("=")[1].split()[0]) * RY, mag=np.nan)
        elif pending is not None and "total magnetization" in L:
            pending["mag"] = float(L.split("=")[1].split()[0])
        elif pending is not None and "Forces acting on atoms" in L:
            f, k = [], i + 1
            while len(f) < nat:
                if "force =" in lines[k]:
                    f.append([float(x) for x in lines[k].split("=")[1].split()[:3]])
                k += 1
            pending["F"] = np.array(f) * RY_BOHR
            frames.append(pending)
            pending, i = None, k
            continue
        elif "Entering Dynamics:" in L:
            it = int(L.split("=")[1])
            t_ps = float(lines[i + 1].split("=")[1].split()[0])
            if frames and "step" not in frames[-1]:
                frames[-1]["step"] = it - 1
            last_it = (it, t_ps)
        elif "temperature           =" in L and last_it is not None:
            thermo.append((last_it[0], last_it[1], float(L.split("=")[1].split()[0])))
        i += 1
    for k, fr in enumerate(frames):            # último SCF sin "Entering Dynamics" detrás
        if "step" not in fr:
            fr["step"] = (frames[k - 1]["step"] if k else (last_it[0] if last_it else 0)) + 1
    thermo = np.array(thermo)
    dt_tiempo = float(np.polyfit(thermo[:, 0], thermo[:, 1], 1)[0]) * 1000 if len(thermo) > 2 else None
    meta = dict(output=Path(path).name, dt_fs_cabecera=dt_fs, dt_fs_tiempo=dt_tiempo, n_scf=len(frames),
                scf_no_convergido=any("convergence NOT achieved" in l for l in lines))
    return dict(meta=meta, species=species, cell=np.array(cell_alat) * alat * BOHR,
                frames=frames, thermo=thermo, last_pos=cur_pos)


def leer_trayectoria(carpeta):
    # Une los output.1, .2, ... de una carpeta en una sola trayectoria ordenada por paso.
    rel = carpeta.relative_to(DATA_ROOT).parts          # ('DATASET_AIMD_LUMI', 'stable', '2000'[, 'case_a'])
    info = dict(hpc=rel[0].replace("DATASET_AIMD_", ""), phase=rel[1], T=int(rel[2]),
                case=rel[3] if len(rel) > 3 else "single")
    info["tag"] = f"{info['hpc']}_{info['phase']}_{info['T']}K_{info['case']}"
    outs = sorted(carpeta.glob("output.*"), key=lambda p: int(p.suffix[1:]))
    frames, thermo, metas = [], [], []
    prev_last_pos = dt = None
    for p in outs:
        o = leer_output(p)
        metas.append(o["meta"])
        if o["meta"]["dt_fs_cabecera"] is not None:
            dt = o["meta"]["dt_fs_cabecera"]             # solo output.1 imprime "Time step"
        if prev_last_pos is not None and o["frames"]:
            o["frames"][0]["pos"] = prev_last_pos.copy() # corrección del reinicio
        prev_last_pos = o["last_pos"]
        frames += [dict(fr, source=p.name) for fr in o["frames"]]
        thermo.append(o["thermo"])
        species, cell = o["species"], o["cell"]
    por_paso, n_dup = {}, 0                              # mismo paso en dos outputs: se queda el primero
    for fr in frames:
        if fr["step"] in por_paso:
            n_dup += 1
            continue
        por_paso[fr["step"]] = fr
    steps = np.array(sorted(por_paso))
    fr = [por_paso[s] for s in steps]
    info.update(dt_fs=dt, n_outputs=len(outs), duplicados=n_dup, huecos=int(np.sum(np.diff(steps) != 1)))
    return dict(info=info, metas=metas, species=np.array(species), cell=cell,
                pos=np.array([f["pos"] for f in fr]), forces=np.array([f["F"] for f in fr]),
                energy=np.array([f["E"] for f in fr]), mag=np.array([f["mag"] for f in fr]),
                step=steps, time_ps=steps * dt / 1000, source=np.array([f["source"] for f in fr]),
                thermo=np.vstack(thermo))

In [ ]:
def tau_int(x):
    # Tiempo de autocorrelación integrado (en pasos): 1 + 2·Σρ(k) hasta el primer cruce por cero.
    x = x - x.mean(); n = len(x)
    f = np.fft.rfft(x, 2 * n)
    ac = np.fft.irfft(f * np.conj(f))[:n]; ac /= ac[0]
    s = 1.0
    for k in range(1, n // 2):
        if ac[k] <= 0:
            break
        s += 2 * ac[k]
    return s


def fps(Y, n):
    # Farthest Point Sampling: empieza por el punto más central y añade siempre el más lejano a los ya elegidos.
    # Devuelve los índices elegidos y, para cada uno, su distancia al elegido más cercano en ese momento.
    n = min(n, len(Y))
    idx, dmax = np.empty(n, int), np.full(n, np.inf)
    idx[0] = int(np.argmin(((Y - Y.mean(0)) ** 2).sum(1)))
    d2 = ((Y - Y[idx[0]]) ** 2).sum(1)
    for k in range(1, n):
        idx[k] = int(np.argmax(d2))
        dmax[k] = np.sqrt(d2[idx[k]])
        np.minimum(d2, ((Y - Y[idx[k]]) ** 2).sum(1), out=d2)
    return idx, dmax


def a_intervalos(t_fs, intervalo_fs):
    # Índices de los frames más cercanos a t0, t0 + Δ, t0 + 2Δ, ... dentro de la trayectoria.
    objetivos = np.arange(t_fs[0], t_fs[-1] + 1e-6, intervalo_fs)
    return np.unique(np.abs(t_fs[:, None] - objetivos[None, :]).argmin(axis=0))


def huella(df):
    # Identificador corto del contenido del dataset, para no reutilizar resultados guardados que ya no corresponden.
    return hashlib.md5(pd.util.hash_pandas_object(df[["traj", "step"]], index=False).values).hexdigest()[:10]

## 3. Leer las simulaciones
Cada carpeta con `output.*` es **una trayectoria**. Las carpetas `case_a/b/c` son trayectorias independientes. La lectura tarda ≈ 1 min la primera vez; después se carga de `WORK`.

**Qué mirar:** deberían salir 18 trayectorias, con `huecos = 0`.

In [ ]:
cache = WORK / "trayectorias.pkl"
if cache.exists() and not RECALCULAR:
    TRAJ = pickle.load(open(cache, "rb"))
    print(f"Cargadas {len(TRAJ)} trayectorias guardadas en {cache}")
else:
    TRAJ = {}
    carpetas = sorted({p.parent for p in DATA_ROOT.glob("DATASET_AIMD_*/**/output.*")})
    for carpeta in carpetas:
        tr = leer_trayectoria(carpeta)
        TRAJ[tr["info"]["tag"]] = tr
        print(f"  {tr['info']['tag']:32s} {len(tr['step']):5d} pasos")
    pickle.dump(TRAJ, open(cache, "wb"))

SPECIES = next(iter(TRAJ.values()))["species"]
CELL    = next(iter(TRAJ.values()))["cell"]
trayectorias = pd.DataFrame([dict(tr["info"], pasos=len(tr["step"]), duracion_ps=round(tr["time_ps"].max(), 2))
                             for tr in TRAJ.values()]).set_index("tag").sort_values(["T", "hpc", "case"])
print(f"{len(SPECIES)} átomos ({(SPECIES == 'C').sum()} C + {(SPECIES == 'B').sum()} B)")
trayectorias[["hpc", "phase", "T", "case", "dt_fs", "n_outputs", "pasos", "duracion_ps", "duplicados", "huecos"]]

## 4. Comprobaciones
* **Paso temporal.** Solo `output.1` escribe `Time step`. Para cada output, incluidos los reinicios, se comprueba con el tiempo que `pw.x` escribe en cada paso.
* **Reinicios.** En cada unión entre outputs, el desplazamiento de los átomos debe ser como el de un paso normal. Si se hubiera usado la geometría de la cabecera, el error sería el de la última columna.

**Qué mirar:** las dos líneas deben terminar en `OK`.

In [ ]:
filas = [dict(trayectoria=tag, output=m["output"], dt_cabecera=tr["info"]["dt_fs"], dt_medido=m["dt_fs_tiempo"])
         for tag, tr in TRAJ.items() for m in tr["metas"] if m["dt_fs_tiempo"] is not None]
dts = pd.DataFrame(filas)
dt_ok = np.allclose(dts["dt_cabecera"], dts["dt_medido"], rtol=1e-3, atol=2e-4)
print(f"Paso temporal: {trayectorias['dt_fs'].value_counts().to_dict()} (fs: nº de trayectorias)."
      f" Coincide con el tiempo impreso en los {len(dts)} outputs: {'OK' if dt_ok else 'REVISAR'}")

filas = []
for tag, tr in TRAJ.items():
    src = tr["source"]
    salto = np.linalg.norm(np.diff(tr["pos"], axis=0), axis=2).max(axis=1)
    for j in np.where(src[1:] != src[:-1])[0]:
        filas.append(dict(trayectoria=tag, union=f"{src[j]} → {src[j + 1]}", salto_en_union_A=salto[j],
                          salto_normal_A=np.median(salto),
                          error_si_cabecera_A=np.linalg.norm(tr["pos"][j + 1] - tr["pos"][0], axis=1).max()))
uniones = pd.DataFrame(filas)
cont_ok = (uniones["salto_en_union_A"] < 3 * uniones["salto_normal_A"]).all()
print(f"Reinicios: {len(uniones)} uniones; todas continuas: {'OK' if cont_ok else 'REVISAR'}")
uniones.round(3)

## 5. Construir el dataset
Para cada trayectoria se aplican, en este orden:
1. Se quitan los primeros `T_EQ_PS` ps (equilibrado). El tiempo se cuenta desde el inicio de la trayectoria, no desde cada reinicio.
2. En las metaestables se quitan los últimos `N_ULTIMOS_METAESTABLE` pasos.
3. Se quitan las configuraciones con algún |F| > `F_MAX`.
4. Solo entran las trayectorias **estables** de `T_ESTABLES` y las **metaestables** de `T_METAESTABLES`.

LUMI y MN5 se tratan igual. Cada configuración se etiqueta con su temperatura (`T`) y su régimen (`phase`: stable / metastable). A partir de aquí, estables y metaestables siguen caminos separados.

In [ ]:
assert sorted(CUOTA_ESTABLES) == sorted(T_ESTABLES), "CUOTA_ESTABLES debe tener una entrada por cada temperatura de T_ESTABLES"

filas, POS, FOR, ENE, resumen = [], [], [], [], []
for tag, tr in TRAJ.items():
    inf = tr["info"]
    fmax = np.linalg.norm(tr["forces"], axis=2).max(axis=1)
    usar = ((inf["phase"] == "stable" and inf["T"] in T_ESTABLES) or
            (inf["phase"] == "metastable" and inf["T"] in T_METAESTABLES))
    ok_eq  = tr["time_ps"] >= T_EQ_PS - 1e-9
    n_fin  = N_ULTIMOS_METAESTABLE if inf["phase"] == "metastable" else 0
    ok_fin = tr["step"] <= tr["step"].max() - n_fin
    ok_f   = fmax <= F_MAX
    keep = ok_eq & ok_fin & ok_f & usar
    resumen.append(dict(traj=tag, T=inf["T"], phase=inf["phase"], usada="sí" if usar else "no", pasos=len(keep),
                        fuera_equilibrado=int((~ok_eq).sum()), fuera_final=int((ok_eq & ~ok_fin).sum()),
                        fuera_fuerza=int((ok_eq & ok_fin & ~ok_f).sum()), en_dataset=int(keep.sum())))
    for k in np.where(keep)[0]:
        filas.append(dict(traj=tag, hpc=inf["hpc"], phase=inf["phase"], T=inf["T"], case=inf["case"],
                          dt_fs=inf["dt_fs"], step=int(tr["step"][k]), time_ps=float(tr["time_ps"][k]),
                          energy_eV=float(tr["energy"][k]), fmax_eV_A=float(fmax[k]), mag_bohr=float(tr["mag"][k])))
    POS.append(tr["pos"][keep]); FOR.append(tr["forces"][keep]); ENE.append(tr["energy"][keep])

frames = pd.DataFrame(filas)
frames.insert(0, "frame_id", np.arange(len(frames)))
POS, FOR, ENE = np.concatenate(POS), np.concatenate(FOR), np.concatenate(ENE)
TEMPS = sorted(int(t) for t in frames["T"].unique())
USADAS = set(frames["traj"])
HUELLA = huella(frames)

display(pd.DataFrame(resumen).set_index("traj").sort_values(["T", "phase"]))
print(f"Configuraciones en el dataset: {len(frames)}")
frames.groupby(["T", "phase"]).size().unstack(fill_value=0)

## 6. Termalización
Temperatura instantánea de cada trayectoria que entra en el dataset. La zona gris es lo que se descarta al principio.

**Qué mirar:** después de la zona gris, la temperatura oscila alrededor de la línea discontinua (la temperatura objetivo).

In [ ]:
nc = 4
fig, axes = plt.subplots(int(np.ceil(len(TEMPS) / nc)), nc, figsize=(13, 5.6), squeeze=False)
for ax, T in zip(axes.flat, TEMPS):
    tmax = 0
    for tr in TRAJ.values():
        if tr["info"]["T"] != T or tr["info"]["tag"] not in USADAS:
            continue
        th = tr["thermo"]
        ax.plot(th[:, 1], th[:, 2], lw=0.7, color=AZUL if tr["info"]["hpc"] == "LUMI" else NARANJA)
        tmax = max(tmax, th[:, 1].max())
    ax.axvspan(0, T_EQ_PS, color="#898781", alpha=0.18, lw=0)
    ax.axhline(T, color=TINTA, lw=1, ls="--")
    ax.set(xlim=(0, max(tmax, 1)), ylim=(0, 1.6 * T), title=f"{T} K", xlabel="tiempo (ps)")
for ax in axes.flat[len(TEMPS):]:
    ax.set_visible(False)
for ax in axes[:, 0]:
    ax.set_ylabel("T instantánea (K)")
fig.legend([Line2D([], [], color=AZUL, lw=2), Line2D([], [], color=NARANJA, lw=2),
            Line2D([], [], color=TINTA, ls="--"), Patch(color="#898781", alpha=0.18)],
           ["LUMI", "MN5", "T objetivo", f"descartado (t < {T_EQ_PS} ps)"], ncol=4, loc="upper right", bbox_to_anchor=(1, 1.04))
fig.tight_layout(); guardar(fig, "01_termalizacion"); plt.show()

## 7. Fuerzas
Izquierda: todas las fuerzas atómicas. Derecha: la fuerza máxima de cada configuración, por temperatura. La línea roja es `F_MAX`.

**Qué mirar:** nada debería acercarse a la línea roja. El máximo es de unos 27 eV/Å.

In [ ]:
FNORM = np.linalg.norm(FOR, axis=2)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw=dict(width_ratios=[1, 1.25]))
a1.hist(FNORM.ravel(), bins=np.arange(0, F_MAX + 6, 1.0), color=AZUL, edgecolor="white", linewidth=0.3)
a1.set_yscale("log"); a1.axvline(F_MAX, color=ROJO, lw=1.6, ls="--")
a1.set(xlim=(0, F_MAX + 5), xlabel="|F| por átomo (eV/Å)", ylabel="nº de átomos (log)", title="Todas las fuerzas")
bp = a2.boxplot([frames.loc[frames["T"] == T, "fmax_eV_A"] for T in TEMPS], positions=range(len(TEMPS)),
                widths=0.55, patch_artist=True, medianprops=dict(color="black"),
                flierprops=dict(marker="o", markersize=2.5, alpha=0.4))
for b in bp["boxes"]:
    b.set(facecolor=AZUL, edgecolor=AZUL, alpha=0.75)
a2.axhline(F_MAX, color=ROJO, lw=1.6, ls="--")
a2.set_xticks(range(len(TEMPS)), TEMPS)
a2.set(ylim=(0, F_MAX + 5), xlabel="temperatura (K)", ylabel="max |F| (eV/Å)", title="Fuerza máxima por configuración")
fig.tight_layout(); guardar(fig, "02_fuerzas"); plt.show()
print(f"Fuerza máxima en todo el dataset: {FNORM.max():.1f} eV/Å")

## 8. Descriptores y UMAP
1. **SOAP** describe el entorno de cada átomo (B y C) hasta 5 Å y se promedia sobre los 126 átomos. El resultado es un vector por configuración, que no cambia si la lámina se desplaza o gira.
2. **Escala y PCA**, ajustadas **solo con las estables**. Comprimen los 390 números de SOAP a unas 35 componentes (99 % de la varianza). Este es el espacio donde FPS y la comprobación de redundancia miden distancias; ajustarlo solo con las estables evita que las metaestables, algunas muy desordenadas, lo deformen. Las metaestables se proyectan en ese mismo espacio únicamente para dibujarlas.
3. **UMAP** proyecta cada configuración en un mapa 2D. Configuraciones parecidas quedan cerca y las islas separadas son configuraciones realmente distintas. El tamaño de las islas y las distancias entre ellas no tienen significado.

In [ ]:
from ase import Atoms
from dscribe.descriptors import SOAP
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import umap

f_soap = WORK / f"soap_{HUELLA}.npy"
if f_soap.exists() and not RECALCULAR:
    X = np.load(f_soap)
else:
    soap = SOAP(species=sorted(set(SPECIES)), periodic=True, average="inner", sparse=False, **SOAP_PARAMS)
    atoms = [Atoms(symbols=list(SPECIES), positions=p, cell=CELL, pbc=True) for p in POS]
    X = soap.create(atoms, n_jobs=N_JOBS).astype(np.float32)
    np.save(f_soap, X)

ES_EST = (frames["phase"] == "stable").values
scaler = StandardScaler().fit(X[ES_EST].astype(np.float64))      # solo estables
Z = scaler.transform(X.astype(np.float64))
pca = PCA(random_state=SEED).fit(Z[ES_EST])                      # solo estables
EVR = pca.explained_variance_ratio_
n_pc = int(np.searchsorted(np.cumsum(EVR), PCA_VAR) + 1)
Y = pca.transform(Z)[:, :n_pc]                                   # estables y metaestables en el mismo espacio

f_umap = WORK / f"umap_v4_{HUELLA}_nn{UMAP_NEIGHBORS}_md{UMAP_MIN_DIST}.npy"
if f_umap.exists() and not RECALCULAR:
    U = np.load(f_umap)
else:
    U = umap.UMAP(n_neighbors=UMAP_NEIGHBORS, min_dist=UMAP_MIN_DIST, random_state=SEED).fit_transform(Y)
    np.save(f_umap, U)
frames["UMAP1"], frames["UMAP2"] = U[:, 0], U[:, 1]
print(f"SOAP: {X.shape[1]} números por configuración  →  PCA (ajustada con las estables): {n_pc} componentes  →  UMAP: 2 coordenadas")

## 9. Mapas UMAP
Cada punto es una configuración.

In [ ]:
orden = np.random.default_rng(SEED).permutation(len(frames))   # orden aleatorio: ningún grupo tapa a otro
d = frames.iloc[orden]
norm = Normalize(min(TEMPS), max(TEMPS))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.scatter(d["UMAP1"], d["UMAP2"], c=d["T"], cmap=HEAT, norm=norm, s=2.5, lw=0, alpha=0.8, rasterized=True)
a1.set_title("Coloreado por temperatura"); limpiar(a1)
cb = fig.colorbar(ScalarMappable(norm=norm, cmap=HEAT), ax=a1, fraction=0.035, pad=0.01)
cb.set_ticks(TEMPS); cb.set_label("temperatura (K)"); cb.outline.set_visible(False)
for fase, col in [("stable", AZUL), ("metastable", NARANJA)]:
    m = d["phase"] == fase
    a2.scatter(d.loc[m, "UMAP1"], d.loc[m, "UMAP2"], s=2.5, lw=0, alpha=0.7, color=col, label=fase, rasterized=True)
a2.set_title("Coloreado por régimen (carpeta)"); limpiar(a2); a2.legend(markerscale=5, loc="best")
fig.tight_layout(); guardar(fig, "03_umap"); plt.show()

Una columna por temperatura: en azul las configuraciones de esa temperatura, en gris el resto.

**Qué mirar:** las metaestables ocupan regiones propias del mapa, separadas de las estables. Por eso en la sección 11 se eligen de forma distinta.

In [ ]:
fig, axes = plt.subplots(1, len(TEMPS), figsize=(2.1 * len(TEMPS), 2.6))
for ax, T in zip(axes, TEMPS):
    ax.scatter(frames["UMAP1"], frames["UMAP2"], s=1, color=GRIS, lw=0, rasterized=True)
    m = frames["T"] == T
    ax.scatter(frames.loc[m, "UMAP1"], frames.loc[m, "UMAP2"], s=2, color=AZUL, lw=0, rasterized=True)
    ax.set_title(f"{T} K", loc="center"); limpiar(ax)
fig.tight_layout(w_pad=0.3); guardar(fig, "04_umap_por_T"); plt.show()

## 10. ¿Cuántas configuraciones independientes hay?
Fotogramas consecutivos de una dinámica son casi iguales. Para cada trayectoria se calcula el tiempo de autocorrelación τ (cuántos pasos hay que esperar para tener una configuración nueva) y

$$n_\text{eff} = \frac{n_\text{fotogramas}}{\tau}$$

n<sub>eff</sub> **no interviene en la selección**. La tabla resumen sirve de guía para elegir `CUOTA_ESTABLES`: una cuota muy por encima de n<sub>eff</sub> añade configuraciones cada vez más parecidas entre sí.

In [ ]:
neff = []
for tag, g in frames.groupby("traj"):
    ii = g.index.values
    tau = np.average([tau_int(Y[ii, k]) for k in range(10)], weights=EVR[:10])
    neff.append(dict(traj=tag, T=g["T"].iloc[0], phase=g["phase"].iloc[0], fotogramas=len(ii),
                     tau_fs=tau * g["dt_fs"].iloc[0], n_eff=len(ii) / tau))
neff = pd.DataFrame(neff).set_index("traj").sort_values(["phase", "T"])
display(neff.round(1))

guia = pd.DataFrame({"disponibles": frames.groupby(["phase", "T"]).size(),
                     "n_eff": neff.groupby(["phase", "T"])["n_eff"].sum().round(0)}).astype(int)
guia["tu cuota (estables)"] = [CUOTA_ESTABLES.get(T, "") if ph == "stable" else "" for ph, T in guia.index]
guia

## 11. Selección de las configuraciones

| Grupo | Temperaturas | Cómo se elige |
|---|---|---|
| **Estables** | `T_ESTABLES` | Para cada temperatura se juntan todas sus trayectorias (case_a/b/c, LUMI y MN5) y se eligen por **FPS** tantas como diga `CUOTA_ESTABLES`. |
| **Metaestables** | `T_METAESTABLES` | En cada trayectoria, una configuración cada `INTERVALO_META_FS` fs. Con `DESCARTAR_ULTIMA_META`, se quita la última (la más cercana al final de la dinámica). |

Después (11.3) se comprueba si alguna estable elegida es **redundante** con otra de distinta temperatura y, si quieres, se quita.

### 11.1 Estables: FPS por temperatura
FPS empieza por la configuración más central y añade siempre la más lejana a las ya elegidas. La **resolución** de cada temperatura es la distancia a la que estaba la última elegida: ninguna configuración disponible de esa temperatura queda más lejos que eso de alguna elegida. Cuanto más pequeña la cuota o más variada la temperatura, mayor la resolución.

In [ ]:
frames["seleccion"] = ""                       # "", "FPS", "tiempo" o "redundante"
RESOL = {}
for T in T_ESTABLES:
    ii = frames.index[(frames["phase"] == "stable") & (frames["T"] == T)].values
    assert CUOTA_ESTABLES[T] <= len(ii), f"{T} K: la cuota ({CUOTA_ESTABLES[T]}) supera las disponibles ({len(ii)})"
    o, dmax = fps(Y[ii], CUOTA_ESTABLES[T])
    frames.loc[ii[o], "seleccion"] = "FPS"
    RESOL[T] = dmax[-1]
pd.DataFrame({"trayectorias juntadas": frames[frames.phase == "stable"].groupby("T")["traj"].nunique(),
              "disponibles": frames[frames.phase == "stable"].groupby("T").size(),
              "cuota": pd.Series(CUOTA_ESTABLES),
              "resolución FPS": pd.Series(RESOL).round(2)}).loc[T_ESTABLES]

### 11.2 Metaestables: una cada `INTERVALO_META_FS` fs

In [ ]:
for tag, g in frames[frames["phase"] == "metastable"].groupby("traj"):
    i_sel = a_intervalos(g["time_ps"].values * 1000, INTERVALO_META_FS)
    if DESCARTAR_ULTIMA_META:
        i_sel = i_sel[:-1]
    frames.loc[g.index[i_sel], "seleccion"] = "tiempo"

meta = frames[frames["phase"] == "metastable"]
meta.groupby("traj").agg(T=("T", "first"), desde_ps=("time_ps", "min"), hasta_ps=("time_ps", "max"),
                         elegidas=("seleccion", lambda s: int((s == "tiempo").sum()))).sort_values("T").round(3)

### 11.3 ¿Hay configuraciones redundantes entre temperaturas?
Dentro de cada temperatura, FPS asegura que dos elegidas nunca están más cerca que su resolución. Pero cada temperatura se elige por separado, así que una configuración de 2000 K puede parecerse mucho a una de 2500 K.

**Criterio.** Una pareja de estables elegidas de temperaturas distintas es **redundante** si su distancia SOAP es menor que la resolución de las dos temperaturas (`UMBRAL_REDUNDANCIA = "fps"`). Es decir: si se hubieran elegido juntas, FPS no habría cogido las dos. También puedes poner un número fijo como umbral.

**Referencia.** Dos fotogramas consecutivos de una misma dinámica (≈ 2 fs) están a la distancia que se imprime abajo. Una pareja por debajo de eso sería prácticamente idéntica.

In [ ]:
est_sel = frames.index[frames["seleccion"] == "FPS"].values
Ts = frames.loc[est_sel, "T"].values
D = np.linalg.norm(Y[est_sel][:, None, :] - Y[est_sel][None, :, :], axis=2)
if UMBRAL_REDUNDANCIA == "fps":
    r = np.array([RESOL[t] for t in Ts])
    umbral = np.minimum(r[:, None], r[None, :])
else:
    umbral = np.full(D.shape, float(UMBRAL_REDUNDANCIA))
i, j = np.triu_indices(len(est_sel), 1)
m = (Ts[i] != Ts[j]) & (D[i, j] < umbral[i, j])
aa, bb, ta, tb = est_sel[i[m]], est_sel[j[m]], Ts[i[m]], Ts[j[m]]
sw = ta > tb                                   # ordenar cada pareja: T_a < T_b
pares = pd.DataFrame({"a": np.where(sw, bb, aa), "b": np.where(sw, aa, bb),
                      "T_a": np.minimum(ta, tb), "T_b": np.maximum(ta, tb),
                      "distancia": D[i, j][m]}).sort_values("distancia").reset_index(drop=True)

D_CONSEC = np.median(np.concatenate([np.linalg.norm(np.diff(Y[g.index.values], axis=0), axis=1)
                                     for _, g in frames[frames["phase"] == "stable"].groupby("traj")]))
print(f"Distancia típica entre dos fotogramas consecutivos: {D_CONSEC:.1f}")
print(f"Parejas redundantes entre temperaturas: {len(pares)}")
print(f"  de ellas, prácticamente idénticas (d < {D_CONSEC:.1f}): {int((pares['distancia'] < D_CONSEC).sum())}")
pd.crosstab(pares["T_a"], pares["T_b"]).rename_axis(index="T", columns="con T") if len(pares) else None

Cada punto es una estable elegida; su altura es la distancia a la elegida más cercana **de otra temperatura**. La raya negra de cada columna es la resolución FPS de esa temperatura y la línea gris discontinua, la distancia entre fotogramas consecutivos. En rojo, las que forman parte de alguna pareja redundante.

In [ ]:
nn = np.where(Ts[:, None] == Ts[None, :], np.inf, D).min(axis=1)
en_par = np.isin(est_sel, np.r_[pares["a"].values, pares["b"].values])
fig, ax = plt.subplots(figsize=(9, 4))
for k, T in enumerate(T_ESTABLES):
    mT = Ts == T
    x = k + np.random.default_rng(k).uniform(-0.28, 0.28, mT.sum())
    ax.scatter(x[~en_par[mT]], nn[mT][~en_par[mT]], s=9, color=AZUL, alpha=0.7, lw=0)
    ax.scatter(x[en_par[mT]], nn[mT][en_par[mT]], s=12, color=ROJO, alpha=0.9, lw=0)
    ax.plot([k - 0.38, k + 0.38], [RESOL[T]] * 2, color="black", lw=2)
ax.axhline(D_CONSEC, color=TINTA, ls="--", lw=1)
ax.set_xticks(range(len(T_ESTABLES)), [f"{T} K" for T in T_ESTABLES]); ax.grid(axis="x", visible=False)
ax.set(ylabel="distancia a la más cercana de otra T", ylim=(0, None), title="Redundancia entre temperaturas (estables elegidas)")
ax.legend([Line2D([], [], marker="o", ls="", color=AZUL), Line2D([], [], marker="o", ls="", color=ROJO),
           Line2D([], [], color="black", lw=2), Line2D([], [], color=TINTA, ls="--")],
          ["no redundante", "redundante", "resolución FPS de su T", "fotogramas consecutivos"], loc="upper left", fontsize=8)
fig.tight_layout(); guardar(fig, "05_redundancia"); plt.show()

### 11.4 Quitar las redundantes y resumen
Con `DESCARTAR_REDUNDANTES = True` se recorren las parejas de la más cercana a la más lejana y, si las dos siguen elegidas, se quita una. Con `DESCARTAR_EN = "mayor_cuota"` se quita de la temperatura que tiene más configuraciones en ese momento; con `"mayor_T"` o `"menor_T"`, de la temperatura más alta o más baja de la pareja. Las quitadas quedan marcadas como `redundante` en el CSV.

In [ ]:
quitadas = []
if DESCARTAR_REDUNDANTES and len(pares):
    activas = set(est_sel)
    n_T = frames.loc[est_sel, "T"].value_counts().to_dict()
    for a, b, Ta, Tb in pares[["a", "b", "T_a", "T_b"]].itertuples(index=False):
        if a in activas and b in activas:
            if DESCARTAR_EN == "mayor_T":
                v = a if Ta > Tb else b
            elif DESCARTAR_EN == "menor_T":
                v = a if Ta < Tb else b
            else:
                v = a if n_T[Ta] >= n_T[Tb] else b
            activas.remove(v)
            n_T[int(frames.at[v, "T"])] -= 1
            quitadas.append(v)
    frames.loc[quitadas, "seleccion"] = "redundante"

es = frames["phase"] == "stable"
tabla = pd.DataFrame({
    "estables (disp.)":     frames[es].groupby("T").size(),
    "metaestables (disp.)": frames[~es].groupby("T").size(),
    "cuota FPS":            pd.Series(CUOTA_ESTABLES),
    "quitadas redundantes": frames[frames["seleccion"] == "redundante"].groupby("T").size(),
    "elegidas FPS":         frames[frames["seleccion"] == "FPS"].groupby("T").size(),
    "elegidas tiempo":      frames[frames["seleccion"] == "tiempo"].groupby("T").size(),
}).reindex(TEMPS).fillna(0).astype(int)
tabla["total elegidas"] = tabla["elegidas FPS"] + tabla["elegidas tiempo"]
tabla.loc["Total"] = tabla.sum()
N_SEL = int(tabla.loc["Total", "total elegidas"])
print(f"Quitadas por redundancia: {len(quitadas)}  |  Estables: {int((frames.seleccion == 'FPS').sum())} (FPS)"
      f"  |  Metaestables: {int((frames.seleccion == 'tiempo').sum())} (cada {INTERVALO_META_FS} fs)  |  Total: {N_SEL}")
tabla

**Qué mirar en las figuras:**
* **Barras:** cuántas configuraciones quedan de cada temperatura y con qué método.
* **Mapa:** las elegidas cubren las regiones estables (azul) y las metaestables (naranja); las cruces rojas son las quitadas por redundancia.
* **Líneas de tiempo:** en cada trayectoria metaestable, las elegidas están igualmente espaciadas en el tiempo.

In [ ]:
fig = plt.figure(figsize=(14, 4.8))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])
a1, a2 = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])

t = tabla.drop("Total")
y = np.arange(len(t))[::-1]
a1.barh(y, t["elegidas FPS"], color=AZUL, height=0.62, label="FPS (estables)")
a1.barh(y, t["elegidas tiempo"], left=t["elegidas FPS"], color=NARANJA, height=0.62, label=f"cada {INTERVALO_META_FS} fs (metaestables)")
for yy, tot in zip(y, t["total elegidas"]):
    a1.text(tot + 3, yy, str(tot), va="center", fontsize=9, color=TINTA)
a1.set_yticks(y, [f"{T} K" for T in t.index]); a1.grid(axis="y", visible=False); a1.set_axisbelow(True)
a1.set(xlabel="configuraciones elegidas", title=f"Selección por temperatura (total {N_SEL})",
       xlim=(0, t["total elegidas"].max() * 1.2))
a1.legend(loc="lower right")

a2.scatter(frames["UMAP1"], frames["UMAP2"], s=1.5, color=GRIS, lw=0, rasterized=True, label="todas")
for met, col, mk in [("FPS", AZUL, "o"), ("tiempo", NARANJA, "o"), ("redundante", ROJO, "x")]:
    m = frames["seleccion"] == met
    if m.any():
        a2.scatter(frames.loc[m, "UMAP1"], frames.loc[m, "UMAP2"], s=12 if mk == "o" else 18, marker=mk, color=col,
                   edgecolor="white" if mk == "o" else None, linewidth=0.4 if mk == "o" else 1.0, label=f"{met} ({m.sum()})")
limpiar(a2); a2.set_title("Elegidas sobre el mapa UMAP"); a2.legend(loc="best", markerscale=1.5)
fig.tight_layout(); guardar(fig, "06_seleccion"); plt.show()

trs = sorted(meta["traj"].unique(), key=lambda s: (int(s.split("_")[2][:-1]), s))
fig, ax = plt.subplots(figsize=(12, 0.45 * len(trs) + 1))
for k, tag in enumerate(trs):
    g = frames[frames["traj"] == tag]
    ax.plot([g["time_ps"].min(), g["time_ps"].max()], [k, k], color=GRIS, lw=6, solid_capstyle="butt")
    s = g[g["seleccion"] == "tiempo"]
    ax.scatter(s["time_ps"], [k] * len(s), color=NARANJA, s=10, zorder=3)
    ax.text(g["time_ps"].max() + 0.03, k, f"{len(s)}", va="center", fontsize=8, color=TINTA)
ax.set_yticks(range(len(trs)), trs); ax.grid(axis="y", visible=False)
ax.set(xlabel="tiempo (ps)", title=f"Metaestables: tramo usado (gris) y configuraciones elegidas, una cada {INTERVALO_META_FS} fs")
fig.tight_layout(); guardar(fig, "07_metaestables_tiempo"); plt.show()

## 12. Exportar
* `BGDY_3x3_seleccion_N<n>.extxyz`: energía (eV), fuerzas (eV/Å) y etiquetas en `atoms.info`: `T_K`, `regime`, `seleccion` (FPS / tiempo), `hpc`, `traj`, `md_step`, `time_ps`… Con `WRAP = True` las posiciones salen envueltas en la celda. Energía y fuerzas no cambian, porque cada átomo solo se traslada por vectores de la red.
* `BGDY_3x3_indice_configuraciones.csv`: todas las configuraciones del dataset, con su posición en UMAP y la columna `seleccion` (`FPS`, `tiempo`, `redundante` o vacía).

In [ ]:
from ase.calculators.singlepoint import SinglePointCalculator
from ase.io import write, read

def a_atoms(i):
    r = frames.iloc[i]
    a = Atoms(symbols=list(SPECIES), positions=POS[i], cell=CELL, pbc=True)
    if WRAP:
        a.wrap()                                         # antes de asignar energía y fuerzas
    a.calc = SinglePointCalculator(a, energy=float(ENE[i]), forces=FOR[i])
    a.info.update(config_type=f"T{r['T']}K", T_K=int(r["T"]), regime=r["phase"], seleccion=r["seleccion"] or "no",
                  hpc=r["hpc"], traj=r["traj"], md_step=int(r["step"]), time_ps=round(float(r["time_ps"]), 5),
                  dt_fs=float(r["dt_fs"]), frame_id=int(r["frame_id"]), total_mag_bohr=float(r["mag_bohr"]))
    return a

idx = frames[frames["seleccion"].isin(["FPS", "tiempo"])].sort_values(["phase", "T", "traj", "step"]).index
f_sel = WORK / f"BGDY_3x3_seleccion_N{len(idx)}.extxyz"
write(f_sel, [a_atoms(i) for i in idx], format="extxyz")
frames.round(6).to_csv(WORK / "BGDY_3x3_indice_configuraciones.csv", index=False)
if EXPORTAR_TODO:
    write(WORK / "BGDY_3x3_AIMD_todo.extxyz", [a_atoms(i) for i in range(len(frames))], format="extxyz")

# Comprobación: releer el archivo
chk = read(f_sel, index=":")
frac = np.array([a.get_scaled_positions(wrap=False) for a in chk])
print(f"{f_sel}: {len(chk)} configuraciones")
print(f"  posiciones dentro de la celda: {'OK' if (frac >= 0).all() and (frac < 1).all() else 'NO'}")
print(f"  por régimen y temperatura: {pd.Series([(a.info['regime'], int(a.info['T_K'])) for a in chk]).value_counts().sort_index().to_dict()}")
print(f"Índice: {WORK / 'BGDY_3x3_indice_configuraciones.csv'}")

In [ ]:
# ---- Opciones de exportación (puedes moverlas a la sección 1) ----
EXPORTAR_JUNTAS   = True    # un extxyz con todas las elegidas (estables + metaestables)
EXPORTAR_SEPARADO = True    # además, un extxyz solo con las estables y otro solo con las metaestables

from ase.calculators.singlepoint import SinglePointCalculator
from ase.io import write, read

def a_atoms(i):
    r = frames.iloc[i]
    a = Atoms(symbols=list(SPECIES), positions=POS[i], cell=CELL, pbc=True)
    if WRAP:
        a.wrap()                                         # antes de asignar energía y fuerzas
    a.calc = SinglePointCalculator(a, energy=float(ENE[i]), forces=FOR[i])
    a.info.update(config_type=f"T{r['T']}K", T_K=int(r["T"]), regime=r["phase"], seleccion=r["seleccion"] or "no",
                  hpc=r["hpc"], traj=r["traj"], md_step=int(r["step"]), time_ps=round(float(r["time_ps"]), 5),
                  dt_fs=float(r["dt_fs"]), frame_id=int(r["frame_id"]), total_mag_bohr=float(r["mag_bohr"]))
    return a

elegidas = frames[frames["seleccion"].isin(["FPS", "tiempo"])].sort_values(["phase", "T", "traj", "step"])
grupos = {}                                              # nombre del archivo -> índices de las configuraciones
if EXPORTAR_JUNTAS:
    grupos["seleccion"] = elegidas.index
if EXPORTAR_SEPARADO:
    grupos["estables"]     = elegidas.index[elegidas["phase"] == "stable"]
    grupos["metaestables"] = elegidas.index[elegidas["phase"] == "metastable"]

archivos = []
for nombre, idx in grupos.items():
    if len(idx) == 0:
        print(f"{nombre}: no hay configuraciones elegidas, no se escribe archivo")
        continue
    f = WORK / f"BGDY_3x3_{nombre}_N{len(idx)}.extxyz"
    write(f, [a_atoms(i) for i in idx], format="extxyz")
    archivos.append(f)
frames.round(6).to_csv(WORK / "BGDY_3x3_indice_configuraciones.csv", index=False)
if globals().get("EXPORTAR_TODO", False):
    write(WORK / "BGDY_3x3_AIMD_todo.extxyz", [a_atoms(i) for i in range(len(frames))], format="extxyz")

# Comprobación: releer cada archivo
for f in archivos:
    chk = read(f, index=":")
    frac = np.array([a.get_scaled_positions(wrap=False) for a in chk])
    dentro = "OK" if (frac >= 0).all() and (frac < 1).all() else "NO"
    por_T = pd.Series([int(a.info["T_K"]) for a in chk]).value_counts().sort_index().to_dict()
    print(f"{f.name}: {len(chk)} configuraciones | posiciones dentro de la celda: {dentro} | por T: {por_T}")
print(f"Índice: {WORK / 'BGDY_3x3_indice_configuraciones.csv'}")